# Reward and admission mechanisms on real stakes

Each mechanism is a Param class: its settings have bounds and documentation, and it offers its curve, the AVAX it would mint for a set of stakes, and a check of the properties it must keep. This notebook compares them on one reference cohort of real stakes, repriced as if every stake started on one evaluation date.

**Status:** v0, unreviewed. Branch `prototype/model-v1`.

**Reference cohort:** every validator period and delegation accepted from 2024-11-19 to 2025-11-19 16:00 UTC, the year before Granite, from `derived.staking-periods`. It stops where the validation lockbox starts, so nothing here reads lockbox data. Stakes are summed by role, whole-day duration, and tenure, because rewards are linear in stake. Admission is judged period by period.

In [1]:
from datetime import datetime, timezone

import holoviews as hv
import panel as pn

from avalanche_research import datasets
from avalanche_research.mechanisms import (
    ACP285Ramp, BoundedHarmonicFrontier, ChainAdmission, ChainReward, QuadraticReward, TenureBoost,
)
from avalanche_research.mechanisms.cohort import group_stakes, load_periods, validator_periods
from avalanche_research.views import mechanism_explorer as views

hv.extension("bokeh")
pn.extension()

In [2]:
periods = load_periods(str(datasets.parquet_path("derived.staking-periods", "staking-periods-2026-09-30")))
cohort = group_stakes(periods)
validators = validator_periods(periods)

SUPPLY_NAVAX = 465_373_500 * 10**9           # supply before the cohort's last stake, 2025-11-19
START = datetime(2026, 12, 22, tzinfo=timezone.utc)  # after the ACP-285 ramp ends

cohort.groupby("role")[["periods", "stake_navax"]].sum().assign(stake_avax=lambda t: t.stake_navax / 1e9).drop(columns="stake_navax")

,periods,stake_avax
role,,
delegator,1233029,5.165182e+08
validator,14215,9.552163e+08


## Annual reward rate by duration

Today's chain pays a rate that rises linearly with duration. After the ACP-285 ramp ends, its minimum is 7.5%. The quadratic curve pays less for short stakes and the same at one year. The tenure boost is shown for a validator with eight completed periods.

In [3]:
quadratic = QuadraticReward()
matched = quadratic.at_equal_issuance(views.repriced(cohort, START), SUPPLY_NAVAX)
mechanisms = [ChainReward(), ACP285Ramp(target_min_rate=0.05, label="ACP-285, 5% target"), TenureBoost(), quadratic, matched]
views.curves_plot(mechanisms, SUPPLY_NAVAX, START, completed_periods=8)

:NdOverlay   [mechanism]
   :Curve   [duration_days]   (annual_rate)

## What each mechanism would mint for the cohort

Issuance is the cost every comparison scores. A mechanism that pays more always looks better on stake unless its cost is counted, which is how the old model's quadratic headline arose. The equal-issuance quadratic changes only the curve's shape.

In [4]:
views.issuance_table(mechanisms, cohort, SUPPLY_NAVAX, START).style.format({"issuance_avax": "{:,.0f}", "change_vs_chain": "{:+.1%}"})

,mechanism,issuance_avax,change_vs_chain
0,Chain today,"11,463,799",+0.0%
1,"ACP-285, 5% target","9,908,812",-13.6%
2,Tenure boost,"12,126,471",+5.8%
3,Quadratic curve,"10,962,480",-4.4%
4,"Quadratic curve, equal issuance","11,463,799",-0.0%


## Admission by duration

Today every validator stake of at least 2,000 AVAX is admitted for any allowed duration. Hash's frontier asks more of short commitments and less of long ones. The table gives the share of the cohort's validator periods, and of their AVAX, that each rule would admit. The cohort's periods all ran at least 14 days, because the 48-hour minimum arrived with Helicon.

In [5]:
rules = [ChainAdmission(), BoundedHarmonicFrontier(), BoundedHarmonicFrontier(p=0.71, s_floor_avax=500, label="Frontier, call settings")]
pn.Column(views.admission_plot(rules), views.admission_table(rules, validators, START).round(3))

Column
    [0] HoloViews(NdOverlay, height=360, sizing_mode='fixed', width=640)
    [1] DataFrame(DataFrame)

In [6]:
{rule.label: rule.check(START) for rule in rules}

{'Chain today': [],
 'Admission frontier': [],
 'Frontier, call settings': ['minimum stake is not strictly convex in duration']}

## Explore

Move a setting and the curve, and the AVAX minted for the cohort, update at once. These views compute directly from the mechanism and the cohort. They start no simulation.

In [7]:
pn.Tabs(
    ("Quadratic", views.explorer(QuadraticReward(), cohort, SUPPLY_NAVAX, START)),
    ("Tenure boost", views.explorer(TenureBoost(), cohort, SUPPLY_NAVAX, START)),
    ("ACP-285", views.explorer(ACP285Ramp(), cohort, SUPPLY_NAVAX, datetime(2026, 11, 6, tzinfo=timezone.utc))),
    ("Admission frontier", views.admission_explorer(BoundedHarmonicFrontier(), validators, START)),
)

Tabs
    [0] Row
        [0] Param(QuadraticReward, parameters=['exponent', 'scale'], show_name=False, width=260)
        [1] ParamFunction(function, _pane=Column, defer_load=False)
    [1] Row
        [0] Param(TenureBoost, parameters=['boost_max', ...], show_name=False, width=260)
        [1] ParamFunction(function, _pane=Column, defer_load=False)
    [2] Row
        [0] Param(ACP285Ramp, parameters=['target_min_rate', ...], show_name=False, width=260)
        [1] ParamFunction(function, _pane=Column, defer_load=False)
    [3] Row
        [0] Param(BoundedHarmonicFrontier, parameters=['s_short_avax', ...], show_name=False, width=260)
        [1] ParamFunction(function, _pane=Column, defer_load=False)